### Block 0 – Install dependencies

In [1]:
!pip install -q openai datasets huggingface_hub scikit-learn tqdm pandas

### Block 1 – Imports, API keys, and OpenAI client

In [2]:
import os
import getpass

from datasets import load_dataset
from huggingface_hub import HfFolder
from openai import OpenAI
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import pandas as pd

# OpenAI API key
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key (sk-...): ")
client = OpenAI()

# Hugging Face token for the gated dataset
hf_token = HfFolder.get_token() or os.getenv("HF_TOKEN")
if not hf_token:
    hf_token = getpass.getpass("Hugging Face token (hf-...): ")


OpenAI API key (sk-...): ··········


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Hugging Face token (hf-...): ··········


### Block 2 – Load FiQA-SA test split (235 examples)

In [3]:
DATASET_ID = "TheFinAI/flare-fiqasa"

# Load all splits using the token
ds_all = load_dataset(DATASET_ID, token=hf_token)

print("Available splits and sizes:")
for split_name in ds_all.keys():
    print(f"  {split_name}: {len(ds_all[split_name])}")

# Use the official test split (235 examples)
EVAL_SPLIT = "test"
dataset = ds_all[EVAL_SPLIT]

print("\nUsing split:", EVAL_SPLIT)
print("Number of examples:", len(dataset))
print("Example 0:", dataset[0])

# Hard check to ensure we are really using 235 examples
assert len(dataset) == 235, f"Expected 235 test examples, got {len(dataset)}"


README.md:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

data/train-00000-of-00001-d0f9b6513e12e0(…):   0%|          | 0.00/100k [00:00<?, ?B/s]

data/test-00000-of-00001-faca082021057ac(…):   0%|          | 0.00/35.8k [00:00<?, ?B/s]

data/valid-00000-of-00001-36997935dc03cb(…):   0%|          | 0.00/29.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/750 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/235 [00:00<?, ? examples/s]

Generating valid split:   0%|          | 0/188 [00:00<?, ? examples/s]

Available splits and sizes:
  train: 750
  test: 235
  valid: 188

Using split: test
Number of examples: 235
Example 0: {'id': 'fiqasa938', 'query': 'What is the sentiment of the following financial post: Positive, Negative, or Neutral?\nText: $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.\nAnswer:', 'answer': 'negative', 'text': '$BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash.', 'choices': ['negative', 'positive', 'neutral'], 'gold': 0}


### Block 3 – Label set and normalization helper

In [4]:
LABELS = ["negative", "neutral", "positive"]

def normalize_prediction(raw: str) -> str:
    """
    Map free-form model output to one of the three labels.
    If no label keyword is found, default to 'neutral'.
    """
    text = (raw or "").strip().lower()
    for label in LABELS:
        if label in text:
            return label
    return "neutral"


### Block 4 – o3 sentiment classifier (responses API)

In [5]:
MODEL_NAME_O3 = "o3"

O3_INSTRUCTIONS = (
    "You are a financial sentiment classifier.\n"
    "Given a single sentence from a financial news article or microblog, "
    "classify its overall sentiment from the perspective of an investor as "
    "exactly one of: negative, neutral, or positive.\n"
    "Respond with only one word: negative, neutral, or positive."
)

def classify_with_o3(sentence: str) -> str:
    """
    Call o3 once and return a normalized 3-way label.
    Uses the Responses API and max_output_tokens.
    """
    response = client.responses.create(
        model=MODEL_NAME_O3,
        instructions=O3_INSTRUCTIONS,
        input=sentence,
        reasoning={"effort": "low"},
        max_output_tokens=16,
    )
    raw = response.output_text or ""
    return normalize_prediction(raw)


### Block 5 – Run evaluation on the 235 test examples

In [6]:
y_true = []
y_pred = []

print(f"Evaluating model: {MODEL_NAME_O3} on {len(dataset)} FiQA-SA test examples...\n")

for example in tqdm(dataset, total=len(dataset)):
    sentence = example["text"]
    true_label = example["answer"].lower().strip()
    pred_label = classify_with_o3(sentence)

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total examples:", len(dataset))
print("Got predictions for:", len(y_pred))


Evaluating model: o3 on 235 FiQA-SA test examples...



100%|██████████| 235/235 [04:54<00:00,  1.25s/it]

Total examples: 235
Got predictions for: 235


### Block 6 – Metrics and error analysis

In [8]:
# Overall accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"\nOverall accuracy: {accuracy:.4f}\n")

print("Classification report:")
print(classification_report(y_true, y_pred, labels=LABELS))

# Build a DataFrame with text / gold / pred
df = pd.DataFrame({
    "text": [ex["text"] for ex in dataset],
    "gold": y_true,
    "pred": y_pred,
})

print("\nFirst 10 predictions:")
print(df.head(10).to_string(index=False))

# Error analysis
errors = df[df["gold"] != df["pred"]]
print(f"\nNumber of errors: {len(errors)}")
print("Some error examples:")
print(errors.head(20).to_string(index=False))



Overall accuracy: 0.0766

Classification report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00        76
     neutral       0.08      1.00      0.14        18
    positive       0.00      0.00      0.00       141

    accuracy                           0.08       235
   macro avg       0.03      0.33      0.05       235
weighted avg       0.01      0.08      0.01       235


First 10 predictions:
                                                                                                                            text     gold    pred
                                                     $BBRY Actually lost .03c per share if U incl VZ as no debt and 3.1 in Cash. negative neutral
                                                                         Legal & General share price: Finance chief to step down negative neutral
                                                                 Kingfisher share price slides on cost to im

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
